# 三模型对比：邻居、距离、邻居+距离

对比三个 softmax 模型在 4 个被试上的 AIC、BIC、NLL 指标。

In [1]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import logsumexp
import glob as glob_module

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from fit_softmax import (
    build_all_maps, parse_csv, build_fitting_steps,
    compute_utilities, centroid_distance, NUM_COLORS,
    is_color_legal, count_effective_colors,
    neg_log_likelihood, neg_log_likelihood_region_only,
)

# 中文字体
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'STHeiti', 'Arial Unicode MS']
elif platform.system() == 'Windows':
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
else:
    plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 120

print('导入完成')

导入完成


In [2]:
# 加载数据
maps = build_all_maps()
data_dir = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'data'))
csv_files = sorted(glob_module.glob(os.path.join(data_dir, 'data_*.csv')))
print(f'找到 {len(csv_files)} 个被试文件')

# 解析所有被试
participants = []
for fp in csv_files:
    name = os.path.basename(fp).replace('data_', '').replace('.csv', '')
    actions = parse_csv(fp)
    steps = build_fitting_steps(actions, maps)
    if steps:
        # 为每个 step 添加 prev_region（上一步选的 region，每个 round 的第一步为 None）
        prev_region = None
        prev_round = None
        for s in steps:
            if s['round'] != prev_round:
                s['prev_region'] = None
                prev_round = s['round']
            else:
                s['prev_region'] = prev_region
            prev_region = s['chosen_region']

        participants.append({'name': name, 'steps': steps, 'n': len(steps)})
        print(f'  {name}: {len(steps)} steps')

找到 19 个被试文件
  0: 348 steps
  031: 409 steps
  24: 341 steps
  311: 336 steps
  Aa: 320 steps
  FLT: 390 steps
  Rex: 343 steps
  Wangyudie: 345 steps
  cmf: 325 steps
  fzm: 284 steps
  lyc: 362 steps
  qhw: 351 steps
  yyq: 328 steps
  zhangcheng: 383 steps
  杨田印盈: 336 steps
  熊雯雯: 373 steps
  程子健: 340 steps
  赵雅菲: 357 steps
  郭lx: 434 steps


In [3]:
# ── 扩展的 utility 计算（支持 beta_prev）──

def compute_utilities_ext(uncolored, regions, adjacency, current_colors, prev_region, params):
    """计算每个未着色 region 的 utility，支持 3 个特征。
    
    params 格式: [beta_n, beta_d, beta_prev]
    - beta_n: 已着色邻居数权重
    - beta_d: 到中心距离权重（取负）
    - beta_prev: 是否是上一步邻居的权重
    """
    beta_n = params[0] if len(params) > 0 else 0.0
    beta_d = params[1] if len(params) > 1 else 0.0
    beta_prev = params[2] if len(params) > 2 else 0.0
    
    utilities = np.empty(len(uncolored))
    for i, rid in enumerate(uncolored):
        colored_nbrs = sum(1 for nb in adjacency[rid] if current_colors[nb] is not None)
        dist = centroid_distance(regions[rid])
        is_prev_nb = 1.0 if (prev_region is not None and rid in adjacency[prev_region]) else 0.0
        utilities[i] = beta_n * colored_nbrs + beta_d * (-dist) + beta_prev * is_prev_nb
    return utilities


def nll_ext(full_params, fitting_steps):
    """通用 NLL 函数，使用 3 参数 utility。"""
    total_nll = 0.0
    for step in fitting_steps:
        utilities = compute_utilities_ext(
            step['uncolored'], step['regions'],
            step['adjacency'], step['current_colors'],
            step['prev_region'], full_params
        )
        chosen_idx = step['uncolored'].index(step['chosen_region'])
        log_p = utilities[chosen_idx] - logsumexp(utilities)
        total_nll -= log_p
    return total_nll


# ── 各模型的 NLL 包装函数 ──

def nll_neighbor_only(params, fitting_steps):
    """仅邻居 (1参数: beta_n)"""
    return nll_ext(np.array([params[0], 0.0, 0.0]), fitting_steps)

def nll_distance_only(params, fitting_steps):
    """仅距离 (1参数: beta_d)"""
    return nll_ext(np.array([0.0, params[0], 0.0]), fitting_steps)

def nll_prev_only(params, fitting_steps):
    """仅前邻 (1参数: beta_prev)"""
    return nll_ext(np.array([0.0, 0.0, params[0]]), fitting_steps)

def nll_neighbor_distance(params, fitting_steps):
    """邻居+距离 (2参数: beta_n, beta_d)"""
    return nll_ext(np.array([params[0], params[1], 0.0]), fitting_steps)

def nll_neighbor_prev(params, fitting_steps):
    """邻居+前邻 (2参数: beta_n, beta_prev)"""
    return nll_ext(np.array([params[0], 0.0, params[1]]), fitting_steps)

def nll_distance_prev(params, fitting_steps):
    """距离+前邻 (2参数: beta_d, beta_prev)"""
    return nll_ext(np.array([0.0, params[0], params[1]]), fitting_steps)

def nll_all_three(params, fitting_steps):
    """邻居+距离+前邻 (3参数: beta_n, beta_d, beta_prev)"""
    return nll_ext(params, fitting_steps)

print('模型函数定义完成')

模型函数定义完成


In [ ]:
# ── 拟合所有模型 ──

results = []

models = [
    # (名称, NLL函数, 初始参数, 参数个数)
    ('随机',         None,                 None,                   0),
    ('仅邻居',       nll_neighbor_only,    np.array([1.0]),        1),
    ('仅距离',       nll_distance_only,    np.array([0.1]),        1),
    ('邻居+距离',    nll_neighbor_distance, np.array([1.0, 0.1]),  2),
]

for p in participants:
    for model_name, nll_func, x0, n_params in models:
        if model_name == '随机':
            nll = sum(-np.log(1.0 / len(step['uncolored'])) for step in p['steps'])
            ll = -nll
            n = p['n']
            aic = -2 * ll
            bic = -2 * ll
        else:
            res = minimize(
                nll_func, x0, args=(p['steps'],),
                method='Nelder-Mead',
                options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8},
            )
            nll = res.fun
            ll = -nll
            n = p['n']
            aic = 2 * n_params - 2 * ll
            bic = n_params * np.log(n) - 2 * ll

        likelihood = np.exp(ll)

        results.append({
            'name': p['name'],
            'model': model_name,
            'nll': nll,
            'll': ll,
            'likelihood': likelihood,
            'aic': aic,
            'bic': bic,
            'n_steps': n,
        })
    
    # 每个被试打印一行汇总
    p_results = [r for r in results if r['name'] == p['name']]
    print(f"\n{'='*90}")
    print(f"被试: {p['name']}")
    print(f"{'模型':<14} {'NLL':>8} {'LL':>9} {'AIC':>9} {'BIC':>9}")
    print(f"{'-'*50}")
    for r in p_results:
        print(f"{r['model']:<12} {r['nll']:>8.2f} {r['ll']:>9.2f} {r['aic']:>9.2f} {r['bic']:>9.2f}")


被试: 0
模型                  NLL        LL       AIC       BIC
--------------------------------------------------
随机             865.79   -865.79   1731.58   1731.58
仅邻居            829.81   -829.81   1661.62   1665.48
仅距离            857.63   -857.63   1717.25   1721.10
仅前邻            844.79   -844.79   1691.59   1695.44
邻居+距离          826.20   -826.20   1656.41   1664.11
邻居+前邻          817.77   -817.77   1639.54   1647.25
距离+前邻          837.38   -837.38   1678.76   1686.46
邻居+距离+前邻       814.06   -814.06   1634.12   1645.68


KeyboardInterrupt: 

In [ ]:
# ── 每个被试的汇总表格 ──
import pandas as pd

all_model_names = ['随机', '仅邻居', '仅距离', 
                   '邻居+距离']

rows = []
for r in results:
    rows.append({
        '被试': r['name'],
        '模型': r['model'],
        '步骤数': r['n_steps'],
        'NLL': f"{r['nll']:.2f}",
        'LL': f"{r['ll']:.2f}",
        'AIC': f"{r['aic']:.2f}",
        'BIC': f"{r['bic']:.2f}",
    })

df = pd.DataFrame(rows)

for name in df['被试'].unique():
    print(f"\n{'='*70}")
    print(f"被试: {name}")
    print(f"{'='*70}")
    sub = df[df['被试'] == name].drop(columns=['被试'])
    print(sub.to_string(index=False))

In [ ]:
# ── 平均每步 likelihood ──
# 每步的 log-likelihood 平均值 = -NLL / n_steps
# 平均每步 likelihood = exp(-NLL / n_steps)
# 随机基线的平均每步 likelihood = exp(-NLL_random / n_steps) ≈ 1/平均可选区域数

import pandas as pd

model_names_ordered = ['随机', '仅邻居', '仅距离',
                       '邻居+距离']

# 计算每个被试每个模型的平均每步 likelihood
rows = []
for r in results:
    avg_ll_per_step = -r['nll'] / r['n_steps']
    avg_likelihood = np.exp(avg_ll_per_step)
    rows.append({
        '被试': r['name'],
        '模型': r['model'],
        'NLL': r['nll'],
        '步骤数': r['n_steps'],
        '平均每步LL': avg_ll_per_step,
        '平均每步likelihood': avg_likelihood,
    })

df_avg = pd.DataFrame(rows)

# 按模型汇总（所有被试平均）
print("各模型的平均每步 likelihood（所有被试平均）")
print(f"{'模型':<14} {'平均每步likelihood':>18} {'平均每步LL':>14} {'随机倍数':>10}")
print('-' * 60)

# 先拿随机基线的均值
random_avg_lk = df_avg[df_avg['模型'] == '随机']['平均每步likelihood'].mean()

for model in model_names_ordered:
    sub = df_avg[df_avg['模型'] == model]
    mean_lk = sub['平均每步likelihood'].mean()
    mean_ll = sub['平均每步LL'].mean()
    ratio = mean_lk / random_avg_lk
    print(f"{model:<12} {mean_lk:>18.4f} {mean_ll:>14.4f} {ratio:>10.2f}x")

# 计算随机基线对应的平均可选区域数
all_uncolored_sizes = []
for p in participants:
    for s in p['steps']:
        all_uncolored_sizes.append(len(s['uncolored']))
avg_choices = np.mean(all_uncolored_sizes)
print(f"\n平均每步可选区域数: {avg_choices:.1f}")
print(f"随机基线平均每步 likelihood ≈ 1/{avg_choices:.1f} = {1/avg_choices:.4f}")

# 每个被试的详细表
print(f"\n{'='*80}")
print("每个被试的平均每步 likelihood")
print(f"{'='*80}")
for name in df_avg['被试'].unique():
    sub = df_avg[df_avg['被试'] == name][['模型', '步骤数', '平均每步likelihood']].copy()
    sub['平均每步likelihood'] = sub['平均每步likelihood'].map(lambda x: f"{x:.4f}")
    print(f"\n{name}:")
    print(sub.to_string(index=False))

In [ ]:
# ── 构建数据 ──

all_model_names = ['随机', '仅邻居', '仅距离', '邻居+距离']
participant_names = [p['name'] for p in participants]
metrics = ['aic', 'bic', 'nll']
metric_labels = ['AIC', 'BIC', 'NLL']

# metric -> model -> [每个被试的值]
data = {}
for metric in metrics:
    data[metric] = {}
    for model in all_model_names:
        data[metric][model] = [r[metric] for r in results if r['model'] == model]

participant_order = sorted(participant_names)
participant_colors = {
    participant: plt.cm.tab20(i % 20) for i, participant in enumerate(participant_order)
}


def align_zero_positions(axes_list, pad=0.05):
    ymins, ymaxs = [], []
    for ax in axes_list:
        ymin, ymax = ax.get_ylim()
        ymins.append(ymin)
        ymaxs.append(ymax)

    max_below = max(abs(min(y, 0)) for y in ymins)
    max_above = max(max(y, 0) for y in ymaxs)

    if max_below > 0:
        max_below *= (1 + pad)
    if max_above > 0:
        max_above *= (1 + pad)

    for ax in axes_list:
        ax.set_ylim(-max_below, max_above)

def make_comparison_figure(baseline_model, title_text, output_name):
    plot_order = ['随机', '仅邻居', '仅距离', '邻居+距离']
    plot_labels = {
        '随机': '随机',
        '仅邻居': '仅邻居',
        '仅距离': '仅距离',
        '邻居+距离': '邻居+距离',
    }
    bar_colors = {
        '随机': '#c8c8c8',
        '仅邻居': '#9db7e5',
        '仅距离': '#f3b07a',
        '邻居+距离': '#f0d36d',
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharex=True)

    for ax, metric, label in zip(axes, metrics, metric_labels):
        baseline_vals = np.array(data[metric][baseline_model])
        diff_data = {m: np.array(data[metric][m]) - baseline_vals for m in plot_order}

        x = np.arange(len(plot_order))
        means = np.array([np.mean(diff_data[m]) for m in plot_order], dtype=float)
        ses = np.array([np.std(diff_data[m], ddof=1) / np.sqrt(len(diff_data[m])) for m in plot_order], dtype=float)

        ax.bar(
            x,
            means,
            yerr=ses,
            capsize=5,
            width=0.68,
            color=[bar_colors[m] for m in plot_order],
            edgecolor='black',
            linewidth=0.8,
            alpha=0.8,
            zorder=1,
        )

        for j, model in enumerate(plot_order):
            vals_by_participant = {
                p['name']: r[metric] - base
                for p, r, base in zip(
                    participants,
                    [rr for rr in results if rr['model'] == model],
                    baseline_vals,
                )
            }
            sub_names = sorted(vals_by_participant)
            offsets = np.linspace(-0.12, 0.12, len(sub_names)) if len(sub_names) > 1 else np.array([0.0])
            for offset, pname in zip(offsets, sub_names):
                ax.scatter(
                    j + offset,
                    vals_by_participant[pname],
                    s=42,
                    color=participant_colors[pname],
                    edgecolor='none',
                    alpha=0.95,
                    zorder=3,
                    label=pname if (label == 'AIC' and model == plot_order[0]) else None,
                )

        ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.8)
        ax.set_title(label, fontsize=20)
        ax.set_ylabel(f'Δ{label}（模型 − {baseline_model}）')
        ax.set_xticks(x)
        ax.set_xticklabels([plot_labels[m] for m in plot_order], rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.25)

    align_zero_positions(axes)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='被试', loc='upper center', ncol=min(10, len(labels)), bbox_to_anchor=(0.5, 0.98))
    fig.suptitle(title_text, y=1.04, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.86])
    fig.savefig(output_name, dpi=300, bbox_inches='tight')
    plt.show()


make_comparison_figure(
    baseline_model='随机',
    title_text='各模型相对于随机模型的差异（bar = 均值 ± SE，点 = 各被试）',
    output_name='model_vs_random.png',
)

make_comparison_figure(
    baseline_model='邻居+距离',
    title_text='各模型相对于邻居+距离模型的差异（bar = 均值 ± SE，点 = 各被试）',
    output_name='model_vs_best.png',
)



In [ ]:
# ── 邻居转移率（Neighbor Transition Rate）──
# 对每个被试的每个 round，统计连续两步中下一步选的 region 是上一步 region 的邻居的比例

def compute_neighbor_transition_rate(steps):
    """计算一个被试所有 round 的邻居转移率。
    
    返回:
        overall_rate: 总体转移率
        per_round: dict {round: rate}
        n_transitions: 总转移次数
        n_neighbor: 其中是邻居的次数
    """
    # 按 round 分组
    from collections import defaultdict
    rounds = defaultdict(list)
    for s in steps:
        rounds[s['round']].append(s)
    
    total_transitions = 0
    total_neighbor = 0
    per_round = {}
    
    for rnd, round_steps in rounds.items():
        n_trans = 0
        n_nb = 0
        for i in range(1, len(round_steps)):
            prev_region = round_steps[i - 1]['chosen_region']
            curr_region = round_steps[i]['chosen_region']
            adjacency = round_steps[i]['adjacency']
            n_trans += 1
            if curr_region in adjacency[prev_region]:
                n_nb += 1
        if n_trans > 0:
            per_round[rnd] = n_nb / n_trans
        total_transitions += n_trans
        total_neighbor += n_nb
    
    overall_rate = total_neighbor / total_transitions if total_transitions > 0 else 0
    return overall_rate, per_round, total_transitions, total_neighbor

# 计算每个被试的邻居转移率
print(f"{'被试':>10} | {'转移率':>8} | {'邻居次数':>8} / {'总次数':>6}")
print('-' * 50)
for p in participants:
    rate, per_round, n_trans, n_nb = compute_neighbor_transition_rate(p['steps'])
    p['neighbor_rate'] = rate
    p['neighbor_per_round'] = per_round
    print(f"{p['name']:>10} | {rate:>8.1%} | {n_nb:>8} / {n_trans:>6}")

# 随机基线：如果随机选择，期望的邻居转移率是多少？
# = 平均每步中，上一步 region 的邻居占未着色 region 的比例
random_rates = []
for p in participants:
    from collections import defaultdict
    rounds = defaultdict(list)
    for s in p['steps']:
        rounds[s['round']].append(s)
    
    total_expected = 0
    total_count = 0
    for rnd, round_steps in rounds.items():
        for i in range(1, len(round_steps)):
            prev_region = round_steps[i - 1]['chosen_region']
            adjacency = round_steps[i]['adjacency']
            uncolored = round_steps[i]['uncolored']
            # 上一步 region 的邻居中有多少在未着色列表里
            nb_in_uncolored = sum(1 for nb in adjacency[prev_region] if nb in uncolored)
            expected_rate = nb_in_uncolored / len(uncolored) if len(uncolored) > 0 else 0
            total_expected += expected_rate
            total_count += 1
    random_rates.append(total_expected / total_count if total_count > 0 else 0)

print(f"\n随机基线（期望邻居转移率）: {np.mean(random_rates):.1%} ± {np.std(random_rates, ddof=1):.1%}")